In [51]:
import os, pickle

from physics.simulation import mcfm, msq
from physics.hzz import zz4l
from datasets import balanced
from models import carl

import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
import hist

import torch
from torch.utils.data import Dataset
import lightning
torch.set_float32_matmul_precision('high')

In [52]:
torch.set_float32_matmul_precision('highest')

matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,  # Don't override with default matplotlib fonts
    "pgf.preamble": "", 
})


In [ ]:
OUTPUT_DIR = 'run/h4l/carl/'
SCALER_FILE_X = 'scaler.pkl'
SAMPLE_DIR = 'data/'

TRAIN_EPOCH = '41'
VAL_LOSS = '0.68'
VERSION = '0'

LOGS_DIR = f'{OUTPUT_DIR}/lightning_logs/version_{VERSION}'
CHECKPOINT_DIR = f'{LOGS_DIR}/checkpoints/'
CHECKPOINT = f'carl-epoch={TRAIN_EPOCH}-val_loss={VAL_LOSS}.ckpt'

FEATURES=['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy',
          'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy',
          'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy',
          'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']

BATCH_SIZE = 1024
SEED = 42

In [54]:
with open(f'{OUTPUT_DIR}/scaler.pkl', 'rb') as f:
    scaler_X = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_numerator_train.pkl', 'rb') as f:
    events_numerator_train = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_denominator_train.pkl', 'rb') as f:
    events_denominator_train = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_numerator_val.pkl', 'rb') as f:
    events_numerator_val = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_denominator_val.pkl', 'rb') as f:
    events_denominator_val = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_numerator_test.pkl', 'rb') as f:
    events_numerator_test = pickle.load(f)
with open(f'{OUTPUT_DIR}/events_denominator_test.pkl', 'rb') as f:
    events_denominator_test = pickle.load(f)

training_data = balanced.BalancedDataset(events_numerator_train, events_denominator_train, FEATURES)
validation_data = balanced.BalancedDataset(events_numerator_val, events_denominator_val, FEATURES)
testing_data = balanced.BalancedDataset(events_numerator_test, events_denominator_test, FEATURES)

training_data.X = scaler_X.transform(training_data.X)
validation_data.X = scaler_X.transform(validation_data.X)
testing_data.X = scaler_X.transform(testing_data.X)

In [55]:
loaded_model = carl.CARL.load_from_checkpoint(os.path.join(CHECKPOINT_DIR, CHECKPOINT))

dl_train = torch.utils.data.DataLoader(training_data, batch_size=BATCH_SIZE)
dl_val = torch.utils.data.DataLoader(validation_data, batch_size=BATCH_SIZE)
dl_test = torch.utils.data.DataLoader(testing_data, batch_size=BATCH_SIZE)

trainer = lightning.Trainer(accelerator='gpu', devices=1)

# predictions_train = torch.concatenate(trainer.predict(loaded_model, dataloaders=[dl_train]), axis=0).detach().numpy()
predictions_val = torch.concatenate(trainer.predict(loaded_model, dataloaders=[dl_val]), axis=0).detach().numpy()
predictions_test = torch.concatenate(trainer.predict(loaded_model, dataloaders=[dl_test]), axis=0).detach().numpy()

# targets_train = training_data.s
targets_val = validation_data.s
targets_test = testing_data.s

FileNotFoundError: [Errno 2] No such file or directory: '/ptmp/mpp/taepa/higgs-offshell-interpretation/run/h4l/carl//version_0/checkpoints/carl-epoch=41-val_loss=0.68.ckpt'

In [ ]:
s_nbins = 50
s_min, s_max = 0.0, 1.0
s_step = (s_max - s_min)/s_nbins

In [ ]:
p = predictions_val
t = targets_val

pred_binned = [p[(p > s_min+s_step*i) & (p <= s_min+s_step*(i+1))] for i in range(s_nbins)]
targets_binned = [t[(p > s_min+s_step*i) & (p <= s_min+s_step*(i+1))] for i in range(s_nbins)]

sig_per_bin = np.array([np.sum(targets_binned[i]==1.0) for i in range(s_nbins)])
bkg_per_bin = np.array([np.sum(targets_binned[i]==0.0) for i in range(s_nbins)])

s_true = sig_per_bin/(sig_per_bin+bkg_per_bin)

sig_err = np.array([np.std(targets_binned[i]==1.0) for i in range(s_nbins)])
bkg_err = np.array([np.std(targets_binned[i]==0.0) for i in range(s_nbins)])
s_err = np.sqrt((sig_err * bkg_per_bin/(sig_per_bin + bkg_per_bin)**2)**2 + (-bkg_err*sig_per_bin/(sig_per_bin + bkg_per_bin)**2)**2)

s_centers = [s_min+(i+1/2)*s_step for i in range(s_nbins)]

fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(4,5), sharex=True)

ax1.errorbar(s_centers, s_centers, color='black', linestyle='--', label='$\mathrm{MC}$ $\mathrm{estimate}$')
ax1.errorbar(s_centers, s_true, yerr=s_err, color='royalblue', marker='o', markersize=4, linestyle='none', label='$\mathrm{NSBI}$ $\mathrm{estimate}$')

ax1.set_ylim(s_min, s_max)
ax1.set_ylabel('$s(x) = \\frac{p_{q\\bar{q}}(x)}{ p_{q\\bar{q}}(x) + p_{gg}(x) }$')

ax1.legend(frameon=False)

ax2.errorbar(s_centers, np.array(s_centers)-np.array(s_centers), color='black', linestyle='--')
ax2.errorbar(s_centers, np.array(s_true)-np.array(s_centers), yerr=0., color='royalblue', marker='o', markersize=4, linestyle='none')

ax2.set_xlim(s_min, s_max)
ax2.set_xlabel('$\\hat{s}(x)$')
ax2.set_ylabel('$\\mathrm{Residual}$')
ax2.set_ylim(-0.5, 0.5)

plt.tight_layout()
plt.savefig('carl_calibration.pdf', bbox_inches='tight')
plt.close()

/tmp/ipykernel_1502326/17646609.py:10: RuntimeWarning: invalid value encountered in divide
  s_true = sig_per_bin/(sig_per_bin+bkg_per_bin)
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/conda/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
